# 💃 Auravo Studio — Free Kaggle GPU Dance Diffusion Worker
### Runs on Kaggle Free GPU (2x T4 or P100 GPU • 73GB Disk Space • 30h Free / Week)

Kaggle is the **recommended free GPU environment** because it provides **73GB disk storage** (solving Google Colab's disk space limitations) and supports **longer 9–12 hour continuous sessions**.

---

### ⚡ 3-Step Setup Instructions for Kaggle:
1. **Enable Internet & GPU in Kaggle Sidebar**:
   - In the right-hand panel under **Settings**:
     - **Internet**: Toggle to **Internet On** *(Required for downloading AI models)*
     - **Accelerator**: Select **GPU T4 x2** (or **GPU P100**)
2. **Run CELL 1** below to install packages (~1-2 minutes).
3. **Run CELL 2** to launch the video diffusion server and start the free Cloudflare tunnel.
4. **Copy the tunnel URL** (`https://xxxx.trycloudflare.com`) printed in the output and paste it into Auravo Studio under **⚙️ Neural AI GPU Settings**!

In [ ]:
# CELL 1: Install Dependencies (Run once per Kaggle session)
!pip install -q fastapi uvicorn pycloudflared nest-asyncio python-multipart torch torchvision torchaudio diffusers transformers accelerate imageio[ffmpeg] opencv-python Pillow requests
print("✅ Kaggle AI Worker dependencies installed successfully!")

In [ ]:
# CELL 2: Start Neural Dance Diffusion Server & Free Cloudflare Tunnel
import os
import gc
import io
import time
import uuid
import base64
import torch
import uvicorn
import nest_asyncio
import threading
from typing import Optional, Dict, Any
from fastapi import FastAPI, HTTPException, Header, Depends, BackgroundTasks, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel
from pycloudflared import try_cloudflare
from PIL import Image, ImageOps

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

app = FastAPI(title="Auravo Neural Diffusion Worker (Kaggle Edition)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

JOBS: Dict[str, Dict[str, Any]] = {}

# Kaggle working directory has 73GB+ disk space
OUTPUT_DIR = "/kaggle/working/storage" if os.path.exists("/kaggle/working") else "/content/storage" if os.path.exists("/content") else os.path.abspath("./storage")
os.makedirs(OUTPUT_DIR, exist_ok=True)

svd_pipeline = None

def get_diffusion_pipeline():
    global svd_pipeline
    if svd_pipeline is None:
        if torch.cuda.is_available():
            print("[Kaggle Worker] Loading Stable Video Diffusion XT onto GPU...")
            from diffusers import StableVideoDiffusionPipeline
            svd_pipeline = StableVideoDiffusionPipeline.from_pretrained(
                "stabilityai/stable-video-diffusion-img2vid-xt",
                torch_dtype=torch.float16,
                variant="fp16"
            ).to("cuda")
            svd_pipeline.enable_vae_slicing()
            svd_pipeline.enable_attention_slicing()
            print("[Kaggle Worker] ✅ Video Diffusion Model ready in GPU memory!")
    return svd_pipeline

def unload_diffusion_pipeline():
    global svd_pipeline
    if svd_pipeline is not None:
        del svd_pipeline
        svd_pipeline = None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        gc.collect()
        print("[Kaggle Worker] 🧹 VRAM cleared after inference.")

def run_diffusion_task(job_id: str, input_image_path: str, prompt: str, duration_sec: int, motion_bucket_id: Optional[int] = None, seed: Optional[int] = None):
    try:
        JOBS[job_id]["status"] = "INITIALIZING"
        JOBS[job_id]["progress"] = 15
        JOBS[job_id]["currentStage"] = "Preprocessing character portrait into 9:16 vertical composition..."

        img = Image.open(input_image_path).convert("RGB")
        # Strict aspect-ratio preserving smart crop to exact 9:16 (zero face distortion/stretching)
        img_resized = ImageOps.fit(img, (576, 1024), method=Image.Resampling.LANCZOS, centering=(0.5, 0.25))

        raw_video_path = os.path.join(OUTPUT_DIR, f"{job_id}_raw.mp4")
        final_video_path = os.path.join(OUTPUT_DIR, f"{job_id}.mp4")

        pipe = get_diffusion_pipeline()

        if pipe is not None and torch.cuda.is_available():
            JOBS[job_id]["status"] = "DIFFUSING"
            JOBS[job_id]["progress"] = 40
            JOBS[job_id]["currentStage"] = "Diffusing neural dance movement & fluid cloth dynamics on GPU..."

            num_frames = 25
            seed_val = int(seed) if seed is not None else (int(time.time() * 1000) % (2**31))
            generator = torch.manual_seed(seed_val)
            mb_id = int(motion_bucket_id) if motion_bucket_id is not None else 140
            with torch.inference_mode():
                frames = pipe(
                    img_resized,
                    decode_chunk_size=4,
                    generator=generator,
                    motion_bucket_id=mb_id,
                    fps=7,
                    num_frames=num_frames
                ).frames[0]

            JOBS[job_id]["status"] = "ENCODING"
            JOBS[job_id]["progress"] = 85
            JOBS[job_id]["currentStage"] = "Encoding fluid 30fps vertical video stream..."

            import imageio
            imageio.mimsave(raw_video_path, frames, fps=12)

            os.system(f"""
            ffmpeg -y -stream_loop 2 -i {raw_video_path} \
              -t {duration_sec} \
              -vf "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,fps=30,format=yuv420p" \
              -c:v libx264 -preset fast -pix_fmt yuv420p {final_video_path} 2>/dev/null
            """)
        else:
            print("[Warning] GPU pipeline unavailable, rendering aspect-preserved 30fps fallback...")
            os.system(f"""
            ffmpeg -y -loop 1 -i {input_image_path} \
              -vf "scale=2160:3840:force_original_aspect_ratio=increase,crop=2160:3840,zoompan=z='min(zoom+0.0007,1.20)':x='(iw-iw/zoom)/2':y='(ih-ih/zoom)*0.25':d=1:s=1080x1920:fps=30,format=yuv420p" \
              -c:v libx264 -preset fast -t {duration_sec} -r 30 {final_video_path} 2>/dev/null
            """)

        unload_diffusion_pipeline()

        JOBS[job_id]["status"] = "COMPLETED"
        JOBS[job_id]["progress"] = 100
        JOBS[job_id]["currentStage"] = "Video generation complete!"
        JOBS[job_id]["videoPath"] = final_video_path
        JOBS[job_id]["videoUrl"] = f"/api/jobs/{job_id}/video.mp4"
    except Exception as e:
        print(f"[Error in Job {job_id}]: {e}")
        JOBS[job_id]["status"] = "FAILED"
        JOBS[job_id]["error"] = str(e)
        unload_diffusion_pipeline()

@app.get("/health")
def health():
    cuda_avail = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if cuda_avail else "CPU"
    device_count = torch.cuda.device_count() if cuda_avail else 0
    vram_mb = round(torch.cuda.memory_allocated(0) / (1024 * 1024), 1) if cuda_avail else 0
    return {
        "status": "ready",
        "gpuAvailable": cuda_avail,
        "device": f"{gpu_name} (x{device_count})" if device_count > 1 else gpu_name,
        "vramAllocatedMB": vram_mb,
        "storage": "73GB+ Kaggle Environment",
        "engine": "Stable Video Diffusion XT / Kaggle T4x2"
    }

@app.post("/api/jobs/dance")
async def create_dance_job(
    background_tasks: BackgroundTasks,
    image: UploadFile = File(...),
    prompt: str = Form("Bollywood expressive classical dance in monsoon rain"),
    durationSec: int = Form(5),
    motionBucketId: Optional[int] = Form(None),
    seed: Optional[int] = Form(None),
    sceneOrder: Optional[int] = Form(None)
):
    job_id = str(uuid.uuid4())[:8]
    input_path = os.path.join(OUTPUT_DIR, f"input_{job_id}.png")
    contents = await image.read()
    with open(input_path, "wb") as f:
        f.write(contents)

    JOBS[job_id] = {
        "id": job_id,
        "status": "QUEUED",
        "progress": 5,
        "currentStage": "Queued on Kaggle GPU diffusion pipeline...",
        "createdAt": time.time(),
        "sceneOrder": sceneOrder,
        "motionBucketId": motionBucketId,
        "seed": seed
    }

    background_tasks.add_task(run_diffusion_task, job_id, input_path, prompt, durationSec, motionBucketId, seed)
    return {"ok": True, "jobId": job_id, "pollUrl": f"/api/jobs/{job_id}"}

@app.get("/api/jobs/{job_id}")
def get_job_status(job_id: str):
    if job_id not in JOBS:
        raise HTTPException(status_code=404, detail="Job not found")
    return JOBS[job_id]

@app.get("/api/jobs/{job_id}/video.mp4")
def stream_job_video(job_id: str):
    video_path = os.path.join(OUTPUT_DIR, f"{job_id}.mp4")
    if not os.path.exists(video_path):
        raise HTTPException(status_code=404, detail="Video file not found or not finished yet")
    return FileResponse(video_path, media_type="video/mp4", filename=f"{job_id}.mp4")

# Start Free Cloudflare Tunnel and Server
os.system("fuser -k 8000/tcp 2>/dev/null || true")
time.sleep(1)
nest_asyncio.apply()
tunnel = try_cloudflare(port=8000)
public_url = getattr(tunnel, "tunnel", str(tunnel))

print("\n" + "="*65)
print("🚀 YOUR FREE KAGGLE NEURAL VIDEO DIFFUSION SERVER IS RUNNING AT:")
print(f"👉  {public_url}  👈")
print("="*65 + "\n")
print("Copy the 👉 trycloudflare.com URL 👈 above and paste it into Auravo Studio!")

print("✅ Kaggle GPU Server active (73GB storage, T4 GPU) waiting for render requests!")
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()
